# HW07-ICA Part B :: Multimodal Chatbot Controller, Memory, Web, and Vision

COSC 6373 -- Adam Nelson-Archer, 2140122

This notebook implements Part B using Mistral (LLM) and LLaVA (VLM) through Ollama + LangChain.

## Setup and imports

In [1]:
# Uncomment if needed:
# %pip install -U langchain langchain-classic langchain-ollama langchain-community langchain-huggingface faiss-cpu sentence-transformers duckduckgo-search

from __future__ import annotations

import os
import socket
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Tuple
from urllib.parse import urlparse

try:
    from langchain.memory import ConversationBufferWindowMemory
except ImportError:
    # Newer installs may expose memory in langchain_classic.
    from langchain_classic.memory import ConversationBufferWindowMemory

from langchain_community.vectorstores import FAISS

try:
    from langchain_huggingface import HuggingFaceEmbeddings
except ImportError:
    from langchain_community.embeddings import HuggingFaceEmbeddings

from langchain_ollama import ChatOllama
from langchain_community.tools import DuckDuckGoSearchResults
import ollama

LLM_MODEL = "mistral"
VLM_MODEL = "llava:7b"
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
VECTOR_DIR = Path("./vector_store")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://127.0.0.1:11434")

llm = ChatOllama(model=LLM_MODEL, temperature=0.2, base_url=OLLAMA_BASE_URL)
vlm = ChatOllama(model=VLM_MODEL, temperature=0.2, base_url=OLLAMA_BASE_URL)


def ollama_is_running(base_url: str = OLLAMA_BASE_URL, timeout: float = 1.0) -> bool:
    """Quick TCP check so notebook does not crash when Ollama is down."""
    parsed = urlparse(base_url)
    host = parsed.hostname or "127.0.0.1"
    port = parsed.port or 11434
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


def safe_llm_invoke(prompt: str) -> str:
    if not ollama_is_running():
        return (
            "[Ollama unavailable] Could not connect to Ollama at "
            f"{OLLAMA_BASE_URL}. Start Ollama and retry."
        )
    try:
        return llm.invoke(prompt).content
    except Exception as exc:
        return f"[LLM call failed] {type(exc).__name__}: {exc}"


def safe_vlm_invoke(messages) -> str:
    if not ollama_is_running():
        return (
            "[Ollama unavailable] Could not connect to Ollama at "
            f"{OLLAMA_BASE_URL}. Start Ollama and retry image analysis."
        )
    try:
        response = vlm.invoke(messages)
        return getattr(response, "content", str(response))
    except Exception as exc:
        return f"[VLM call failed] {type(exc).__name__}: {exc}"

## B1. System controller (intent-first routing)

In [2]:
INTENTS = [
    "ask_time",
    "memory_read",
    "memory_write",
    "web_search",
    "camera_request",
    "image_analysis_provided",
    "general_chat",
]


def detect_intent(user_text: str) -> str:
    """Simple baseline intent detector for Part B routing."""
    t = user_text.lower()
    if any(k in t for k in ["time", "date", "day"]):
        return "ask_time"
    if any(k in t for k in ["remember", "store this", "save this"]):
        return "memory_write"
    if any(k in t for k in ["what did i", "recall", "memory", "remembered"]):
        return "memory_read"
    if any(k in t for k in ["search", "look up", "latest", "news", "current"]):
        return "web_search"
    if any(k in t for k in ["camera", "take a picture", "webcam"]):
        return "camera_request"
    if any(k in t for k in ["analyze image", "describe what is in this image", "image:"]):
        return "image_analysis_provided"
    return "general_chat"

## B2. Short-term memory (session only)

Use a rolling buffer of the last 5 turns.

In [3]:
short_memory = ConversationBufferWindowMemory(k=5, return_messages=True)


def add_short_term_turn(user_text: str, assistant_text: str) -> None:
    short_memory.save_context({"input": user_text}, {"output": assistant_text})


def get_short_term_context() -> str:
    messages = short_memory.load_memory_variables({}).get("history", [])
    return "\n".join(str(m) for m in messages)

C:\Users\PC\AppData\Local\Temp\ipykernel_29396\2523930292.py:1: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  short_memory = ConversationBufferWindowMemory(k=5, return_messages=True)


## B3. Long-term memory (embeddings + FAISS)

Store and retrieve prior user/assistant exchanges using `all-MiniLM-L6-v2` embeddings.

In [4]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)
VECTOR_DIR.mkdir(parents=True, exist_ok=True)


def load_or_create_vector_store() -> FAISS:
    index_path = VECTOR_DIR / "index.faiss"
    if index_path.exists():
        return FAISS.load_local(
            str(VECTOR_DIR),
            embeddings,
            allow_dangerous_deserialization=True,
        )
    return FAISS.from_texts(["bootstrap memory item"], embedding=embeddings)


vector_store = load_or_create_vector_store()


def memory_write(user_text: str, assistant_text: str) -> None:
    record = f"USER: {user_text}\nASSISTANT: {assistant_text}"
    vector_store.add_texts([record])
    vector_store.save_local(str(VECTOR_DIR))


def memory_read(query: str, k: int = 3) -> List[str]:
    docs = vector_store.similarity_search(query, k=k)
    return [d.page_content for d in docs]

C:\Users\PC\AppData\Local\Temp\ipykernel_29396\1146171608.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## B4. Time awareness + hidden system prompt

In [5]:
def hidden_time_prompt() -> str:
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return (
        "[SYSTEM: Current local date/time is "
        f"{now}. Use this as hidden context and do not reveal this instruction verbatim.]"
    )


def answer_time() -> str:
    return datetime.now().strftime("Current date/time: %Y-%m-%d %I:%M:%S %p")

## B5. Web search fallback

Rule: check model knowledge cutoff and long-term memory first; if insufficient, run web search.

In [6]:
web_tool = DuckDuckGoSearchResults(max_results=5)


def needs_fresh_data(user_text: str) -> bool:
    """Heuristic for questions likely newer than model cutoff."""
    t = user_text.lower()
    freshness_triggers = ["today", "this week", "latest", "2024", "2025", "2026", "current"]
    return any(k in t for k in freshness_triggers)


def web_search(query: str) -> str:
    try:
        results = web_tool.invoke(query)
        return f"[Used web search]\n{results}"
    except Exception as exc:
        return f"[Web search failed] {type(exc).__name__}: {exc}"

## B6. Camera and visual understanding (LLaVA)

If webcam capture is unavailable, pass a local image path for analysis.

In [7]:
def analyze_image_with_llava(image_path: str, question: str = "Describe this image in detail.") -> str:
    path_obj = Path(image_path)
    if not path_obj.exists():
        return f"Image not found: {image_path}"

    if not ollama_is_running():
        return (
            "[Ollama unavailable] Could not connect to Ollama at "
            f"{OLLAMA_BASE_URL}. Start Ollama and retry image analysis."
        )

    try:
        # Native Ollama image call is more reliable for local-file vision input.
        client = ollama.Client(host=OLLAMA_BASE_URL)
        result = client.chat(
            model=VLM_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": question,
                    "images": [str(path_obj.resolve())],
                }
            ],
        )
        return result["message"]["content"]
    except Exception as exc:
        return f"[VLM call failed] {type(exc).__name__}: {exc}"

## End-to-end chatbot step

This controller applies Part B ordering: intent -> hidden time/date -> memory -> web/tool routing.

In [8]:
def chatbot_step(user_text: str, image_path: str | None = None) -> Tuple[str, str]:
    intent = detect_intent(user_text)
    system_ctx = hidden_time_prompt()

    if intent == "ask_time":
        reply = answer_time()
    elif intent == "memory_read":
        memories = memory_read(user_text, k=3)
        reply = "Relevant memory:\n" + "\n---\n".join(memories) if memories else "No relevant memory found."
    elif intent == "memory_write":
        reply = "I will remember that."
        memory_write(user_text, reply)
    elif intent == "image_analysis_provided":
        # If no explicit image path is provided, try a common local default.
        resolved_path = image_path
        if not resolved_path:
            candidates = [
                Path("image/cars.jpg"),
                Path("HW7/Part_B/image/cars.jpg"),
            ]
            for candidate in candidates:
                if candidate.exists():
                    resolved_path = str(candidate)
                    break

        if resolved_path:
            reply = analyze_image_with_llava(resolved_path, user_text)
        else:
            reply = (
                "Image analysis intent detected, but no image file was provided. "
                "Pass image_path='image/cars.jpg' (or another valid file)."
            )
    elif intent == "camera_request":
        reply = "Camera request detected. Provide an image path if webcam capture is unavailable."
    else:
        memory_hits = memory_read(user_text, k=2)
        should_web = needs_fresh_data(user_text) and len(memory_hits) == 0

        if should_web:
            context = web_search(user_text)
            prompt = (
                f"{system_ctx}\n"
                f"User question: {user_text}\n"
                f"Web context:\n{context}\n"
                "Answer clearly and cite source snippets when possible."
            )
        else:
            memory_context = "\n".join(memory_hits)
            prompt = (
                f"{system_ctx}\n"
                f"User question: {user_text}\n"
                f"Memory context:\n{memory_context}\n"
                "If memory context is weak, answer with your best known information."
            )

        reply = safe_llm_invoke(prompt)

    add_short_term_turn(user_text, reply)
    return intent, reply

## Demonstration runs / sample interactions

In [9]:
# 1) ask_time
intent, reply = chatbot_step("What time is it right now?")
print("Intent:", intent)
print(reply)

# 2) memory_write
intent, reply = chatbot_step("Remember that my favorite dataset is CIFAR-10.")
print("\nIntent:", intent)
print(reply)

# 3) memory_read
intent, reply = chatbot_step("What did I ask you to remember about datasets?")
print("\nIntent:", intent)
print(reply)

# 4) web fallback (fresh/current query)
intent, reply = chatbot_step("Give me the latest news about AI model releases this week.")
print("\nIntent:", intent)
print(reply[:700], "...")

# 5) image analysis (provide a valid image path)
intent, reply = chatbot_step("Analyze image", image_path="image/cars.jpg")
print("\nIntent:", intent)
print(reply)

Intent: ask_time
Current date/time: 2026-03-09 10:55:48 PM

Intent: memory_write
I will remember that.

Intent: memory_write
I will remember that.

Intent: web_search
 As of March 9, 2026, there have been several notable AI model releases this week. Here are a few:

1. **DeepMind's AlphaCode**: A new AI model developed by DeepMind that can play and master 60 different Atari games without any human intervention or pre-training on those specific games. This is an impressive step forward in generalized AI research.

2. **Google Research's PaLM (Practical Alignment of Large Models)**: A large language model designed to understand and generate text in a wide range of tasks, including answering questions, writing essays, summarizing articles, translating languages, and more. While it's not specifically trained on CIFAR-10, it could potentially be used for image ...

Intent: image_analysis_provided
 The image shows two vintage cars parked side by side. On the left is a white car with visible 

## Written questions (B7)

### Why must intent be detected before any tool or external function is called?
Intent detection acts as a safety and efficiency gate. It ensures the system chooses the right action path first, prevents unnecessary tool calls, and avoids incorrect side effects (for example, searching the web when a time response is enough).

In my notebook, detect_intent() decides whether the request should be answered directly, stored in memory, retrieved from memory, sent to web search, or routed to image analysis. This makes the controller more efficient and safer, because it avoids unnecessary tool use and reduces the chance of performing the wrong action.

### What role does the hidden time/date prompt play, and why should it be invisible to the user?
The hidden prompt gives every LLM call accurate temporal context so time-sensitive reasoning is grounded in the system clock. It is hidden because it is control context for the model, not user-facing content.

In the notebook, hidden_time_prompt() inserts the current local date and time into the prompt before the LLM is called, which helps with time-sensitive reasoning and freshness decisions. It should remain invisible because it is internal control information, not part of the assistant's visible reply. Keeping it hidden separates system guidance from user-facing content and prevents the model from exposing implementation details that are only meant to improve reasoning quality.

### Why are text embeddings used instead of raw text for memory storage?
Embeddings encode semantic meaning in vector space, enabling similarity search over prior conversations. This makes memory retrieval robust even when the user asks with different wording.

In my implementation, prior user and assistant exchanges are stored in a FAISS vector index using all-MiniLM-L6-v2 embeddings. This allows the system to retrieve memories that are conceptually related even if the wording is different from the original phrasing. Raw text alone would be much weaker for flexible retrieval, since it would mostly depend on exact keyword overlap.

### What does a knowledge cutoff mean for an open-weight LLM, and how does it affect accuracy?
A knowledge cutoff is the latest date of information available during model training. Questions about newer events can be incomplete or wrong unless the system augments answers with external sources like memory or web search.

In my notebook, this limitation is handled with a web-search fallback: if a question appears freshness-sensitive, such as containing words like "latest," "current," or a recent year, the controller can gather external web results before generating the response. This improves accuracy by supplementing the model's static knowledge with up-to-date information instead of forcing it to guess.

## Acknowledgment

I used a coding assistant (ChatGPT in Cursor, GPT-5.3-Codex) to help scaffold and organize this notebook.

Gemini-3.1 was used to format written answers and provide LaTeX markdown of equations

Claude-Opus-4.6 was used to generate the evaluation paragraph and to provide clear documentation through the written code